In [5]:
import fastf1
import fastf1.plotting
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import scipy.stats as stats
import plotly.express as px


In [6]:
fastf1.Cache.enable_cache("cache")

In [7]:
session = fastf1.get_session(2023, 'Bahrain', 'Q')
session.load()

core           INFO 	Loading data for Bahrain Grand Prix - Qualifying [v3.6.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '16', '55', '14', '63', '44', '18', '31', '27', '4', '77', '24', '22', '23', '2', '20', '81', '21', '10']


In [8]:
laps = session.laps
laps.shape

(254, 31)

In [9]:
ver_laps = laps.pick_driver("VER")
ver_laps[["LapNumber", "LapTime", "Sector1Time", "Sector2Time", "Sector3Time"]].head()


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/fastf1/core.py:3183: FutureWarning: pick_driver is deprecated and will be removed in a future release. Use pick_drivers instead.
  warnings.warn(("pick_driver is deprecated and will be removed"


,LapNumber,LapTime,Sector1Time,Sector2Time,Sector3Time
0,1.0,NaT,NaT,NaT,NaT
1,2.0,NaT,NaT,0 days 00:00:53.666000,0 days 00:00:38.509000
2,3.0,0 days 00:01:31.295000,0 days 00:00:29.152000,0 days 00:00:39.195000,0 days 00:00:22.948000
3,4.0,0 days 00:01:49.812000,0 days 00:00:35.615000,0 days 00:00:44.953000,0 days 00:00:29.244000
4,5.0,NaT,NaT,0 days 00:00:53.390000,0 days 00:00:45.107000


In [10]:
fastest_laps = laps.pick_fastest()
fastest_laps[["Driver", "LapTime", "Compound"]]

Driver                         VER
LapTime     0 days 00:01:29.708000
Compound                      SOFT
dtype: object

In [11]:
clean_laps = laps.pick_quicklaps()
df = clean_laps.copy()
df["LapTimeSec"] = df["LapTime"].dt.total_seconds()

fig = px.histogram(
    df,
    x="LapTimeSec",
    color="Driver",
    nbins=20,
    barmode="overlay",
    opacity=0.35,
    labels={"LapTimeSec": "Lap Time (sec)"},
    title="Lap Time Distributions - Bahrain Q 2023",
)
fig.update_layout(template="plotly_white")
fig.show()


In [13]:
clean_laps = laps.pick_quicklaps().copy()
clean_laps["Phase"] = pd.cut(
    clean_laps["LapNumber"],
    bins=[0,16,32,60],
    labels=["Q1", "Q2", "Q3"],
)
clean_laps[["Driver", "LapNumber", "LapTime", "Phase"]].head(10)

,Driver,LapNumber,LapTime,Phase
2,VER,3.0,0 days 00:01:31.295000,Q1
7,VER,8.0,0 days 00:01:30.503000,Q1
10,VER,11.0,0 days 00:01:29.897000,Q1
13,VER,14.0,0 days 00:01:29.708000,Q1
17,PER,3.0,0 days 00:01:31.479000,Q1
22,PER,8.0,0 days 00:01:30.746000,Q1
25,PER,11.0,0 days 00:01:30.131000,Q1
28,PER,14.0,0 days 00:01:29.846000,Q1
34,LEC,5.0,0 days 00:01:31.094000,Q1
39,LEC,10.0,0 days 00:01:31.699000,Q1


In [14]:
def driver_features(df):
    return (
        df.groupby("Driver")
        .agg(
            fastest_lap=("LapTime", "min"),
            median_lap=("LapTime", "median"),
            lap_count=("LapTime", "size"),
            lap_var=("LapTime", lambda s: np.var(s.dt.total_seconds())),
            deleted_laps=("Deleted", "sum"),
        )
        .reset_index()
    )

features_df = driver_features(clean_laps)
features_df["pace_rank"] = features_df["fastest_lap"].rank()
features_df.sort_values("fastest_lap").head()


,Driver,fastest_lap,median_lap,lap_count,lap_var,deleted_laps,pace_rank
18,VER,0 days 00:01:29.708000,0 days 00:01:30.200000,4,0.383451,0,1.0
11,PER,0 days 00:01:29.846000,0 days 00:01:30.438500,4,0.393158,0,2.0
7,LEC,0 days 00:01:30,0 days 00:01:30.688000,4,0.449764,0,3.0
14,SAI,0 days 00:01:30.154000,0 days 00:01:30.754000,6,0.592379,1,4.0
1,ALO,0 days 00:01:30.336000,0 days 00:01:31.094000,5,0.118920,0,5.0


In [18]:
# Requires Team column; compute fastest lap gap to teammate
fast_by_team = (
    clean_laps.groupby(["Team", "Driver"])["LapTime"].min().reset_index()
)
fast_by_team["team_min"] = fast_by_team.groupby("Team")["LapTime"].transform("min")
fast_by_team["delta_to_teammate"] = (
    fast_by_team["LapTime"] - fast_by_team["team_min"]
)
fast_by_team.sort_values("delta_to_teammate",ascending=False).head(10)

,Team,Driver,LapTime,team_min,delta_to_teammate
11,Haas F1 Team,MAG,0 days 00:01:31.892000,0 days 00:01:30.809000,0 days 00:00:01.083000
4,Alpine,GAS,0 days 00:01:31.818000,0 days 00:01:30.914000,0 days 00:00:00.904000
2,AlphaTauri,DEV,0 days 00:01:32.121000,0 days 00:01:31.400000,0 days 00:00:00.721000
13,McLaren,PIA,0 days 00:01:32.101000,0 days 00:01:31.381000,0 days 00:00:00.720000
7,Aston Martin,STR,0 days 00:01:30.836000,0 days 00:01:30.336000,0 days 00:00:00.500000
19,Williams,SAR,0 days 00:01:31.652000,0 days 00:01:31.461000,0 days 00:00:00.191000
9,Ferrari,SAI,0 days 00:01:30.154000,0 days 00:01:30,0 days 00:00:00.154000
16,Red Bull Racing,PER,0 days 00:01:29.846000,0 days 00:01:29.708000,0 days 00:00:00.138000
14,Mercedes,HAM,0 days 00:01:30.384000,0 days 00:01:30.340000,0 days 00:00:00.044000
1,Alfa Romeo,ZHO,0 days 00:01:31.473000,0 days 00:01:31.443000,0 days 00:00:00.030000


In [21]:
concerning = fast_by_team[fast_by_team["delta_to_teammate"] > pd.Timedelta("0.5s")]
concerning

,Team,Driver,LapTime,team_min,delta_to_teammate
2,AlphaTauri,DEV,0 days 00:01:32.121000,0 days 00:01:31.400000,0 days 00:00:00.721000
4,Alpine,GAS,0 days 00:01:31.818000,0 days 00:01:30.914000,0 days 00:00:00.904000
11,Haas F1 Team,MAG,0 days 00:01:31.892000,0 days 00:01:30.809000,0 days 00:00:01.083000
13,McLaren,PIA,0 days 00:01:32.101000,0 days 00:01:31.381000,0 days 00:00:00.720000


In [22]:
summary = (
    fast_by_team.merge(
        clean_laps.groupby("Driver")["LapTime"].size().reset_index(name="quicklap_count"),
        on="Driver",
        how="left",
    )
    .groupby("Team")
    .agg(
        max_gap=("delta_to_teammate", "max"),
        min_gap=("delta_to_teammate", "min"),
        drivers=("Driver", "nunique"),
    )
)
summary


,max_gap,min_gap,drivers
Team,,,
Alfa Romeo,0 days 00:00:00.030000,0 days,2
AlphaTauri,0 days 00:00:00.721000,0 days,2
Alpine,0 days 00:00:00.904000,0 days,2
Aston Martin,0 days 00:00:00.500000,0 days,2
Ferrari,0 days 00:00:00.154000,0 days,2
Haas F1 Team,0 days 00:00:01.083000,0 days,2
McLaren,0 days 00:00:00.720000,0 days,2
Mercedes,0 days 00:00:00.044000,0 days,2
Red Bull Racing,0 days 00:00:00.138000,0 days,2


In [23]:
compound_check = clean_laps.loc[
    clean_laps["Driver"].isin(fast_by_team["Driver"]),
    ["Driver", "Team", "Compound"]
].drop_duplicates()
compound_check.head()


,Driver,Team,Compound
2,VER,Red Bull Racing,SOFT
17,PER,Red Bull Racing,SOFT
34,LEC,Ferrari,SOFT
48,SAI,Ferrari,MEDIUM
51,SAI,Ferrari,SOFT
